We now generate a Vitessce configuration that combines the OME-Zarr image, the OME-Zarr segmentation mask, and the annotated AnnData table into an interactive visualization layout.

For this, we use [`harpy_vitessce`](https://github.com/vibspatial/harpy_vitessce), a small wrapper library around [`vitessce-python`](https://github.com/vitessce/vitessce-python) that provides sensible defaults for image-based and sequence-based transcriptomics datasets, as well as proteomics datasets.

You can explore the [`harpy_vitessce` documentation](https://harpy-vitessce.readthedocs.io/en/latest/) and a few example visualizations created with it: [CODEX proteomics](https://vib-data-core.github.io/vitessce/?url=spatial-hackathon-public/sparrow/public_datasets/proteomics/codex/chl_maps_dataset_vitessce/codex.json) and [Visium HD transcriptomics](https://vib-data-core.github.io/vitessce/?url=spatial-hackathon-public/sparrow/public_datasets/transcriptomics/visium_hd/config_visium_hd_benchmark_s3_10_2_26.json).

Create the environment for this notebook with:

```bash
cd vitessce
uv sync --python 3.12 --locked
source .venv/bin/activate
```

In [1]:
from pathlib import Path

import harpy_vitessce as hpv

BASE_DIR = Path("/Users/arne.defauw/VIB/DATA/test_data/vitessce")

output_path_image = BASE_DIR / "image.ome.zarr"
output_path_mask = BASE_DIR / "mask.ome.zarr"
output_path_adata = BASE_DIR / "adata.zarr"


scale_yx = [0.2125, 0.2125]
translation_yx = [18000, 10000]

coordinate_transformations_image = [
    {"type": "translation", "translation": [0.0, *translation_yx]},
    {"type": "scale", "scale": [1.0, *scale_yx]},
]

coordinate_transformations_mask = [
    {"type": "translation", "translation": translation_yx},
    {"type": "scale", "scale": scale_yx},
]

vc = hpv.imgbased_transcriptomics_from_split_sources(
    img_source="image.ome.zarr",
    labels_source="mask.ome.zarr",
    adata_source="adata.zarr",
    adata_as_spots=True,
    channels=[0, 1, 2, 3],
    spot_radius_size_micron=3,
    base_dir=BASE_DIR,
    coordinate_transformations_image=coordinate_transformations_image,
    coordinate_transformations_mask=coordinate_transformations_mask,
    segmentation_color="#00E5A8",
    segmentation_filled=False,
    embedding_key="X_umap",
    cluster_key="leiden",
    visualize_heatmap=False,
    segmentation_stroke_width=0.005,
    visualize_feature_matrix=True,
    spatial_key="spatial_micron",
)

/Users/arne.defauw/VIB/targeted_transcriptomics_training/.venv_harpy_vitessce/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/Users/arne.defauw/VIB/targeted_transcriptomics_training/.venv_harpy_vitessce/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
2026-03-09 08:50:57.709 | WARNING  | harpy_vitessce.vitessce_config._utils:_validate_camera:28 - zoom was provided without center. Vitessce i

In [3]:
from IPython.display import HTML, display

url = vc.web_app()
display(HTML(f'<a href="{url}" target="_blank">Open in Vitessce</a>'))